# Feeling of Success Split Analysis

This notebook builds the single train/eval/test split used for model training and evaluation. Visually challenging objects do **not** get separate train/eval/test split files; instead, they are tagged as a subset inside the existing splits using `possibly-visually-difficult.csv` as the source of truth.

In [ ]:
from pathlib import Path

# Input manifest generated by scripts/convert_h5_to_png_samples.py.
MANIFEST_CSV = Path("../feeling-of-success/manifest.csv")
VISUALLY_DIFFICULT_CSV = Path("../possibly-visually-difficult.csv")
OUTPUT_DIR = Path("./")

# for testing
# MANIFEST_CSV = Path("../dataset-limited/manifest.csv")
# OUTPUT_DIR = Path("../dataset-limited")

In [ ]:
# Tunable notebook settings.
SEED = 40
SPLIT_RATIOS = {"train": 0.70, "eval": 0.15, "test": 0.15}
SPLIT_GROUP_STRATEGY = "object_type"  # choose "object_type" or "object_type_dataset"

# Set True when you want this notebook to write train/eval/test CSV files.
WRITE_SPLIT_CSVS = True

In [ ]:
import csv
import random
import re
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)

FINAL_MANIFEST_FIELDNAMES = [
    "episode_id",
    "dataset_name",
    "task_id",
    "task_name",
    "object_id",
    "object_class",
    "condition_class",
    "clip_type",
    "result",
    "failure_mode",
    "rgb_path",
    "tactile_path",
    "force_path",
    "metadata_path",
    "split_group",
]


def parse_bool(value):
    normalized = str(value).strip().lower()
    if normalized in {"true", "1", "yes", "success"}:
        return True
    if normalized in {"false", "0", "no", "failure"}:
        return False
    raise ValueError(f"Could not parse boolean value: {value!r}")


def h5_number(source_h5):
    match = re.search(r"_(\d+)$", Path(source_h5).stem)
    if not match:
        raise ValueError(f"Could not find trailing _<number> in source_h5: {source_h5}")
    return match.group(1)


def split_group_for_df(df):
    if SPLIT_GROUP_STRATEGY == "object_type":
        return df["object_type"]
    if SPLIT_GROUP_STRATEGY == "object_type_dataset":
        return df["object_type"].astype(str) + "_" + df["episode_id"].astype(str)
    raise ValueError(f"Unknown split group strategy: {SPLIT_GROUP_STRATEGY}")


def load_visually_challenging_object_types(csv_path):
    visual_df = pd.read_csv(csv_path)
    if "object_name" not in visual_df.columns:
        raise ValueError(f"{csv_path} is missing required column: object_name")
    object_names = sorted(visual_df["object_name"].dropna().astype(str).unique())
    return object_names, visual_df

## Load Metadata And Visually Challenging Object List

In [ ]:
required_columns = {
    "sample_id",
    "source_h5",
    "source_index",
    "object_type",
    "is_gripping",
    "metadata_path",
    "rgb_path",
    "tactile_path",
}

manifest_df = pd.read_csv(MANIFEST_CSV)
missing_columns = required_columns.difference(manifest_df.columns)
if missing_columns:
    raise ValueError(f"Missing required column(s): {sorted(missing_columns)}")

manifest_df = manifest_df.copy()
manifest_df["source_index"] = manifest_df["source_index"].astype(int)
manifest_df["is_gripping"] = manifest_df["is_gripping"].map(parse_bool)
manifest_df["episode_id"] = manifest_df["source_h5"].map(h5_number)
manifest_df["split_group"] = split_group_for_df(manifest_df)

visually_challenging_objects, visually_challenging_source_df = load_visually_challenging_object_types(VISUALLY_DIFFICULT_CSV)
visually_challenging_set = set(visually_challenging_objects)

available_object_types = set(manifest_df["object_type"])
missing_visually_challenging = sorted(visually_challenging_set - available_object_types)
if missing_visually_challenging:
    raise ValueError(f"Visually challenging object types not found in dataset: {missing_visually_challenging}")

manifest_df["is_visually_challenging"] = manifest_df["object_type"].isin(visually_challenging_set)

success_count = int(manifest_df["is_gripping"].sum())
dataset_summary = pd.DataFrame([{
    "samples": len(manifest_df),
    "object_types": manifest_df["object_type"].nunique(),
    "files": manifest_df["episode_id"].nunique(),
    "success": success_count,
    "failure": len(manifest_df) - success_count,
    "success_rate": round(success_count / len(manifest_df), 4),
    "visually_challenging_source_objects": len(visually_challenging_set),
    "visually_challenging_objects_in_manifest": manifest_df.loc[manifest_df["is_visually_challenging"], "object_type"].nunique(),
    "visually_challenging_samples": int(manifest_df["is_visually_challenging"].sum()),
}])

display(dataset_summary)
display(visually_challenging_source_df.head(20))

## Assign Train, Eval, And Test Splits

In [ ]:
ratio_sum = sum(SPLIT_RATIOS.values())
if any(value < 0 for value in SPLIT_RATIOS.values()) or abs(ratio_sum - 1.0) > 1e-9:
    raise ValueError(f"Split ratios must be non-negative and sum to 1.0; got {ratio_sum}")

split_group_table = (
    manifest_df.groupby("split_group", as_index=False)
    .agg(
        object_type=("object_type", "first"),
        episode_id=("episode_id", "first"),
        is_gripping_success_count=("is_gripping", "sum"),
        total_samples=("sample_id", "count"),
        source_h5_files=("source_h5", lambda values: ";".join(sorted({Path(value).name for value in values}))),
    )
)
if SPLIT_GROUP_STRATEGY == "object_type":
    split_group_table["episode_id"] = ""
split_group_table["is_gripping_success_count"] = split_group_table["is_gripping_success_count"].astype(int)
split_group_table["is_gripping_failure_count"] = split_group_table["total_samples"] - split_group_table["is_gripping_success_count"]
split_group_table = split_group_table[
    ["split_group", "object_type", "episode_id", "is_gripping_success_count", "is_gripping_failure_count", "total_samples", "source_h5_files"]
].sort_values("split_group")

sizes = sorted(split_group_table["total_samples"].tolist())
split_group_summary = pd.DataFrame([{
    "strategy": SPLIT_GROUP_STRATEGY,
    "split_groups": len(split_group_table),
    "total_samples": int(sum(sizes)),
    "largest_group": int(max(sizes)),
    "median_group_size": int(sizes[len(sizes) // 2]),
}])
display(split_group_summary)
display(split_group_table.head(30))

shuffled_records = split_group_table.to_dict("records")
random.Random(SEED).shuffle(shuffled_records)
shuffled_groups = pd.DataFrame(shuffled_records)
total_samples = int(shuffled_groups["total_samples"].sum())
train_cutoff = total_samples * SPLIT_RATIOS["train"]
eval_cutoff = train_cutoff + total_samples * SPLIT_RATIOS["eval"]

assigned_splits = []
assigned_samples = 0
for total in shuffled_groups["total_samples"]:
    if assigned_samples < train_cutoff:
        split = "train"
    elif assigned_samples < eval_cutoff:
        split = "eval"
    else:
        split = "test"
    assigned_splits.append(split)
    assigned_samples += int(total)

assigned_group_table = shuffled_groups.assign(split=assigned_splits).sort_values("split_group").reset_index(drop=True)
split_lookup = dict(zip(assigned_group_table["split_group"], assigned_group_table["split"]))
manifest_df["split"] = manifest_df["split_group"].map(split_lookup)

assignment_summary = (
    assigned_group_table.groupby("split", as_index=False)
    .agg(assigned_groups=("split_group", "count"), assigned_samples=("total_samples", "sum"))
)
assignment_summary["target_fraction"] = assignment_summary["split"].map(SPLIT_RATIOS)
assignment_summary["target_samples"] = assignment_summary["target_fraction"].mul(total_samples).round().astype(int)
assignment_summary["assigned_fraction"] = (assignment_summary["assigned_samples"] / total_samples).round(4)
assignment_summary = assignment_summary[["split", "target_fraction", "target_samples", "assigned_samples", "assigned_fraction", "assigned_groups"]]
display(assignment_summary)

## Build Final Manifest Rows

In [ ]:
final_manifest_df = pd.DataFrame({
    "episode_id": manifest_df["episode_id"],
    "dataset_name": "feeling_of_success",
    "task_id": manifest_df["source_index"],
    "task_name": "grasp_outcome",
    "object_id": manifest_df["object_type"],
    "object_class": manifest_df["is_visually_challenging"].map({True: "visually_challenging", False: "normal"}),
    "condition_class": "",
    "clip_type": "before-during-after-images",
    "result": manifest_df["is_gripping"],
    "failure_mode": manifest_df["is_gripping"].map({True: "", False: "unknown"}),
    "rgb_path": manifest_df["rgb_path"],
    "tactile_path": manifest_df["tactile_path"],
    "force_path": "",
    "metadata_path": manifest_df["metadata_path"],
    "split_group": manifest_df["split_group"],
    "split": manifest_df["split"],
})

row_split_summary = (
    final_manifest_df.groupby("split", as_index=False)
    .agg(
        rows=("task_id", "count"),
        success=("result", "sum"),
        split_groups=("split_group", "nunique"),
        object_types=("object_id", "nunique"),
    )
)
row_split_summary["success"] = row_split_summary["success"].astype(int)
row_split_summary["failure"] = row_split_summary["rows"] - row_split_summary["success"]
row_split_summary["row_fraction"] = (row_split_summary["rows"] / len(final_manifest_df)).round(4)
row_split_summary = row_split_summary[["split", "rows", "success", "failure", "row_fraction", "split_groups", "object_types"]]
display(row_split_summary)

## Visually Challenging Objects Within Existing Splits

These tables answer the new question: given the normal train/eval/test split, how many visually challenging objects and samples are in each split?

In [ ]:
split_visual_summary = (
    final_manifest_df.assign(is_visually_challenging=final_manifest_df["object_class"].eq("visually_challenging"))
    .groupby("split", as_index=False)
    .agg(
        total_samples=("task_id", "count"),
        total_objects=("object_id", "nunique"),
        visually_challenging_samples=("is_visually_challenging", "sum"),
        visually_challenging_objects=("object_id", lambda s: s[final_manifest_df.loc[s.index, "object_class"].eq("visually_challenging")].nunique()),
        visually_challenging_success=("result", lambda s: int(s[final_manifest_df.loc[s.index, "object_class"].eq("visually_challenging")].sum())),
    )
)
split_visual_summary["visually_challenging_samples"] = split_visual_summary["visually_challenging_samples"].astype(int)
split_visual_summary["visually_challenging_failure"] = split_visual_summary["visually_challenging_samples"] - split_visual_summary["visually_challenging_success"]
split_visual_summary["normal_samples"] = split_visual_summary["total_samples"] - split_visual_summary["visually_challenging_samples"]
split_visual_summary["visually_challenging_sample_fraction"] = (
    split_visual_summary["visually_challenging_samples"] / split_visual_summary["total_samples"]
).round(4)
display(split_visual_summary)

visually_challenging_by_split = (
    final_manifest_df[final_manifest_df["object_class"].eq("visually_challenging")]
    .groupby(["split", "object_id"], as_index=False)
    .agg(samples=("task_id", "count"), success=("result", "sum"))
)
visually_challenging_by_split["success"] = visually_challenging_by_split["success"].astype(int)
visually_challenging_by_split["failure"] = visually_challenging_by_split["samples"] - visually_challenging_by_split["success"]
visually_challenging_by_split = visually_challenging_by_split.sort_values(["split", "samples", "object_id"], ascending=[True, False, True])
display(visually_challenging_by_split)

## CSV Export

In [ ]:
def write_final_split_csvs(source_df, filenames_by_split):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for split_name, filename in filenames_by_split.items():
        output_path = OUTPUT_DIR / filename
        split_df = source_df[source_df["split"].eq(split_name)][FINAL_MANIFEST_FIELDNAMES]
        split_df.to_csv(output_path, index=False, quoting=csv.QUOTE_MINIMAL)
        print(f"Wrote {len(split_df):,} rows to {output_path}")


if WRITE_SPLIT_CSVS:
    write_final_split_csvs(final_manifest_df, {
        "train": "train.csv",
        "eval": "eval.csv",
        "test": "test.csv",
    })
else:
    print("WRITE_SPLIT_CSVS is False; train/eval/test split CSV files not written.")

print("Visually challenging examples are tagged in object_class inside train/eval/test; no separate visually challenging split CSVs are written.")